# CRISPR Screen Analysis with MAGeCK

**Tier 3 — Applied Bioinformatics | Module 33 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 14 (Genetic Engineering In Silico)*

---

**By the end of this notebook you will be able to:**
1. Explain pooled CRISPR screen design: positive/negative selection, sgRNA library
2. Process screen data: FASTQ → sgRNA count table with MAGeCK count
3. Run MAGeCK test for gene essentiality scoring (RRA algorithm)
4. Interpret log fold-change, p-value, and FDR for hit calling
5. Visualize screen results: volcano plot, rank plot, sgRNA-level scatter


**Key resources:**
- [MAGeCK documentation](https://sourceforge.net/p/mageck/wiki/Home/)
- [CRISPRScreenAnalyzeR](https://www.crisprscan.org/)
- [DepMap portal](https://depmap.org/portal/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: Screen QC & Normalization →](./02_screen_qc_normalization.ipynb)

---

## 1. Pooled CRISPR Screen Design

> Genome-wide vs targeted sgRNA libraries (Brunello, GeCKO). Positive selection (drug resistance) vs negative selection (growth/viability). Replicate design, library coverage (300–500× per sgRNA). Lentiviral delivery and MOI.

## 2. sgRNA Counting with MAGeCK count

> Trim reads to sgRNA length. Exact match counting. Count table format: sgRNA ID, gene, count per sample. Unmapped read fraction as quality indicator.

In [ ]:
# Example: MAGeCK count
# !mageck count -l library.csv -n sample \
#   --sample-label 'plasmid,day14_rep1,day14_rep2' \
#   --fastq plasmid.fastq.gz day14_rep1.fastq.gz day14_rep2.fastq.gz

## 3. MAGeCK test: Robust Rank Aggregation

> RRA algorithm: rank sgRNAs by LFC, aggregate rankings per gene. Beta score for gene essentiality. Negative binomial model for p-value estimation.

In [ ]:
# Example: MAGeCK test
# !mageck test -k sample.count.txt -t day14_rep1,day14_rep2 -c plasmid \
#   -n mageck_output --gene-lfc-method median

## 4. Hit Calling and Interpretation

> FDR threshold for positive and negative selection hits. Compare to DepMap CERES/Chronos essentiality scores. Enrichment of essential gene categories (proteasome, spliceosome, ribosome).

## 5. Visualization

> Volcano plot (LFC vs -log10 FDR). Rank plot highlighting known essential genes. sgRNA-level scatter plot for top hits. Gini index for library uniformity.